In [ ]:
import pandas as pd
import numpy as np
import tools


In [ ]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_hard_flagged_wazone.parquet")


In [ ]:
# ============================================================
# Pełny kalendarz TowId x Data — BEZ agregacji i bez dodatkowych kolumn.
#
# Zasada:
#   - bierzemy TowId, które miały choć jedną sprzedaż,
#   - dla każdego dnia z zakresu wstawiamy jego RZECZYWISTE rekordy sprzedaży
#     (jeśli tego dnia było kilka transakcji — zostaje kilka wierszy),
#   - jeśli danego dnia nie było żadnej sprzedaży — wstawiamy JEDEN pusty
#     rekord (DokId = -1, reszta kolumn transakcyjnych pusta).
# ============================================================

df_sprzedaz = df_kalendarz[df_kalendarz['TypRuchu'] == 'sprzedaz'].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

# 1. Siatka TowId x Data
siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

# 2. Left merge z surowymi rekordami sprzedaży — bez żadnej agregacji.
#    Dni bez sprzedaży dostają NaN we wszystkich kolumnach transakcyjnych.
pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")

# 3. Atrybuty TowId (te same dla wszystkich dni danego produktu) —
#    uzupełniamy je tam, gdzie merge zostawił puste (dni bez sprzedaży),
#    bo to nie są nowe kolumny, tylko istniejące, wymagające wypełnienia.
kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort',
                  'NazwaTowCleanName', 'NazwaAsortCleanName']
if 'JestWazony' in df_kalendarz.columns:
    kolumny_towid.append('JestWazony')
if 'JestMartwy' in df_kalendarz.columns:
    kolumny_towid.append('JestMartwy')

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

# 4. Sentinel -1 w DokId oznacza "pusty rekord" (brak sprzedaży tego dnia)
pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)

print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


In [ ]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


In [ ]:
pelny_kalendarz.to_parquet(
    "dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [ ]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
